# Explore Activity Data 🏃

**Goal of this notebook:** before we design any dashboard screens, we need to see what
data Garmin Connect actually gives us for a run. This notebook:

1. Logs in to Garmin Connect (reusing the cached session token, just like `example.py`).
2. Finds today's activity (e.g. the interval workout you created in the Garmin Connect app).
3. Pulls the raw data for that activity: the summary, the lap/split data, and the
   "typed splits" (which show structured workout steps like warmup / interval / recovery).
4. Searches through that raw data for the running-dynamics fields we care about:
   **pace, cadence, ground contact time (GCT), vertical oscillation**, etc.
5. Saves the raw JSON to `your_data/` so we can refer back to real field names while
   building the dashboard (this folder is git-ignored, so your data never gets committed).

Nothing here builds any UI yet — it's purely "detective work" so the dashboard we build
next is based on real data, not guesses.

## 1. Connect to Garmin

We reuse the same login approach as `example.py`: read `EMAIL` / `PASSWORD` from the
`.env` file, and cache the session token in `~/.garminconnect` so we don't have to log
in every time we run this notebook.

In [ ]:
import json
import os
import sys
from datetime import date
from pathlib import Path

# Let this notebook import the local `garminconnect` package no matter where
# Jupyter's working directory happens to be (repo root, notebooks/, etc).
REPO_ROOT = Path.cwd() if (Path.cwd() / "garminconnect").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

from garminconnect import (
    Garmin,
    GarminConnectAuthenticationError,
    GarminConnectConnectionError,
    GarminConnectTooManyRequestsError,
)

# Where we'll save raw JSON responses so we can inspect real field names later.
DATA_DIR = REPO_ROOT / "your_data"
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
def get_garmin_client() -> Garmin:
    """Log in to Garmin Connect, reusing a cached token if we have one."""
    tokenstore = Path(os.getenv("GARMINTOKENS", "~/.garminconnect")).expanduser()

    try:
        client = Garmin()
        client.login(str(tokenstore))
        print("✅ Logged in using saved tokens.")
        return client
    except GarminConnectTooManyRequestsError as err:
        raise SystemExit(f"Rate limited by Garmin, try again later: {err}") from err
    except (GarminConnectAuthenticationError, GarminConnectConnectionError):
        print("No valid saved tokens — logging in with email/password from .env ...")

    email = os.getenv("EMAIL")
    password = os.getenv("PASSWORD")
    if not email or not password:
        raise SystemExit("Set EMAIL and PASSWORD in your .env file first.")

    client = Garmin(email=email, password=password)
    client.login(str(tokenstore))
    print(f"✅ Login successful. Tokens saved to: {tokenstore}")
    return client


api = get_garmin_client()
print("Logged in as:", api.get_full_name())

In [ ]:
import pandas as pd


def save_raw(name: str, data: object) -> Path:
    """Save a raw API response as pretty-printed JSON under your_data/, for reference."""
    out_path = DATA_DIR / f"{name}.json"
    out_path.write_text(json.dumps(data, indent=2, default=str))
    print(f"💾 Saved {name} -> {out_path.relative_to(REPO_ROOT)}")
    return out_path


def to_lap_frame(data: object) -> pd.DataFrame:
    """Turn a splits/typed-splits response into a per-lap pandas table.

    Garmin's split endpoints normally return a dict shaped like
    ``{"lapDTOs": [ {...one lap...}, {...next lap...} ]}``. We look for that key
    first, but fall back to scanning for *any* list-of-dicts in the response so
    this keeps working even if Garmin names the field differently for your
    account/activity type.
    """
    if isinstance(data, list):
        return pd.json_normalize(data)

    if isinstance(data, dict):
        if isinstance(data.get("lapDTOs"), list):
            return pd.json_normalize(data["lapDTOs"])
        for value in data.values():
            if isinstance(value, list) and value and isinstance(value[0], dict):
                return pd.json_normalize(value)

    print("⚠️ Could not find a list of laps automatically — inspect the raw JSON above.")
    return pd.DataFrame()


def find_fields(data: object, keywords: list[str], path: str = "") -> list[tuple[str, object]]:
    """Recursively search a nested dict/list for keys containing any of `keywords`.

    Returns a list of (path, example_value) tuples. Useful for hunting down fields
    like "pace", "cadence", "groundContact", "verticalOscillation", etc. without
    knowing the exact key name Garmin uses.
    """
    matches: list[tuple[str, object]] = []
    keywords_lower = [k.lower() for k in keywords]

    if isinstance(data, dict):
        for key, value in data.items():
            new_path = f"{path}.{key}" if path else key
            if any(kw in key.lower() for kw in keywords_lower):
                matches.append((new_path, value))
            matches.extend(find_fields(value, keywords, new_path))
    elif isinstance(data, list):
        for item in data[:1]:  # one example element is enough to see the shape
            matches.extend(find_fields(item, keywords, f"{path}[0]"))

    return matches

## 2. Find today's activity

`get_activities_by_date(start, end)` returns every activity recorded between two dates
(inclusive). We ask for just today, which will include the workout you created in the
Garmin Connect app once it's synced from your watch.

In [ ]:
today = date.today().isoformat()
todays_activities = api.get_activities_by_date(today, today)

print(f"Found {len(todays_activities)} activities today ({today}).\n")

for activity in todays_activities:
    print(
        f"- id={activity['activityId']} | "
        f"'{activity['activityName']}' | "
        f"type={activity['activityType']['typeKey']} | "
        f"start={activity['startTimeLocal']} | "
        f"distance={activity.get('distance', 0) / 1000:.2f} km | "
        f"duration={activity.get('duration', 0) / 60:.1f} min"
    )

if not todays_activities:
    print(
        "No activities found for today yet. If you just finished the workout, "
        "make sure it has synced to Garmin Connect (via the app or a Bluetooth sync), "
        "then re-run this cell."
    )

## 3. Pull the full summary for the latest activity

Garmin returns activities newest-first, so `todays_activities[0]` is today's most
recent activity. We fetch its full summary with `get_activity()`, which includes
overall stats (distance, average pace, average cadence, etc. — the whole-run
averages, not per-interval).

If you have more than one activity today, change `ACTIVITY_INDEX` below to pick a
different one (0 = most recent).

In [ ]:
ACTIVITY_INDEX = 0  # 0 = today's most recent activity

if not todays_activities:
    raise SystemExit("No activities today yet — nothing to explore.")

activity_id = todays_activities[ACTIVITY_INDEX]["activityId"]
activity_summary = api.get_activity(activity_id)
save_raw("latest_activity_summary", activity_summary)

print(f"Exploring activity {activity_id}: '{activity_summary.get('activityName')}'\n")
print(json.dumps(activity_summary, indent=2, default=str))

## 4. Look at the lap/split data (this is where intervals live)

`get_activity_splits()` returns one entry per lap. For a workout you built in the
Garmin Connect app (warmup / interval / recovery / cooldown), each lap usually
corresponds to one step of that workout — this is how we'll identify "the interval
sections" for the dashboard.

Let's dump the raw JSON first, then load it into a pandas table so it's easier to
scan for useful columns (pace, cadence, GCT, vertical oscillation, heart rate...).

In [ ]:
activity_splits = api.get_activity_splits(activity_id)
save_raw("latest_activity_splits", activity_splits)
print(json.dumps(activity_splits, indent=2, default=str))

In [ ]:
laps_df = to_lap_frame(activity_splits)

# Show only columns that actually have data for at least one lap, so the table
# isn't cluttered with empty/irrelevant fields.
laps_df = laps_df.dropna(axis=1, how="all")
laps_df

## 5. Typed splits — structured workout step labels

Because this activity was created as a structured workout, `get_activity_typed_splits()`
should tell us the *type* of each split (e.g. `WARMUP`, `INTERVAL`, `RECOVERY`,
`COOLDOWN`), which is exactly what we need to group laps into "the interval section"
vs "the recovery section" vs "the warmup" on the dashboard.

In [ ]:
activity_typed_splits = api.get_activity_typed_splits(activity_id)
save_raw("latest_activity_typed_splits", activity_typed_splits)
print(json.dumps(activity_typed_splits, indent=2, default=str))

In [ ]:
typed_laps_df = to_lap_frame(activity_typed_splits)
typed_laps_df = typed_laps_df.dropna(axis=1, how="all")
typed_laps_df

## 6. Hunt for the running-dynamics fields we actually care about

Rather than guessing exact Garmin field names, let's search everything we've pulled so
far for keys that look related to **pace, cadence, ground contact time (GCT),
vertical oscillation, stride length**, and **heart rate**. This tells us exactly which
field to use for each stat on the dashboard.

In [ ]:
KEYWORDS = [
    "pace",
    "speed",
    "cadence",
    "groundcontact",
    "ground_contact",
    "verticaloscillation",
    "verticalratio",
    "stride",
    "heartrate",
    "hr",
]

all_sources = {
    "activity_summary": activity_summary,
    "activity_splits": activity_splits,
    "activity_typed_splits": activity_typed_splits,
}

for source_name, source_data in all_sources.items():
    print(f"\n=== {source_name} ===")
    hits = find_fields(source_data, KEYWORDS)
    if not hits:
        print("  (no matching fields found)")
    for field_path, example_value in hits:
        print(f"  {field_path} = {example_value!r}")

## Next steps

Now that we can see the real field names Garmin gives us, here's what's next:

1. **Review the output above** — check the `lapDTOs` tables in sections 4 & 5, and the
   field hits in section 6. Confirm which lap corresponds to which part of your workout
   (warmup / interval / recovery / cooldown), and which exact field names hold pace,
   cadence, GCT, and vertical oscillation for your watch model (these can vary slightly
   between Garmin devices).
2. **Check `your_data/`** — the raw JSON files saved by this notebook
   (`latest_activity_summary.json`, `latest_activity_splits.json`,
   `latest_activity_typed_splits.json`) are there for reference any time, without
   needing to re-run API calls.
3. Once we've confirmed the fields, we'll design the dashboard: a small local web app
   (Streamlit is a great beginner-friendly choice for a Python data dashboard) that
   shows your latest run with a summary card up top and a per-interval breakdown table
   below, using the exact field names we found here.